<a href="https://colab.research.google.com/github/JaysonGaa/cobble-llm/blob/main/cobble_finetune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers datasets peft trl bitsandbytes accelerate


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "codellama/CodeLlama-7b-hf"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

print("Model has been loaded successfully")


In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('cobble_token'))

print("Login Successful")

In [ ]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

model_name = "codellama/CodeLlama-7b-hf"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

print("Model loaded in 4-bit successfully")

In [ ]:
# Test base model
inputs = tokenizer("Fix this Python bug:\ndef add(a, b)\n    return a + b", return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=200)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
from datasets import load_dataset

# First Dataset are coding instruction pairs
dataset1 = load_dataset("sahil2801/CodeAlpaca-20k")

# Second Dataset has coding instructions
dataset2 = load_dataset("ise-uiuc/Magicoder-OSS-Instruct-75K")

print(dataset1)
print(dataset2)

In [ ]:
# Look at a few examples from each dataset
print("=== CodeAlpaca Example ===")
print(dataset1['train'][0])

print("\n=== Magicoder Example ===")
print(dataset2['train'][0])

In [3]:
from datasets import load_dataset

dataset = load_dataset("ise-uiuc/Magicoder-OSS-Instruct-75K")

print(dataset)
print("\n=== Example 1 ===")
print(dataset['train'][0])

print("\n=== Example 2 ===")
print(dataset['train'][1])

print("\n=== Example 3 ===")
print(dataset['train'][2])

DatasetDict({
    train: Dataset({
        features: ['lang', 'raw_index', 'index', 'seed', 'openai_fingerprint', 'problem', 'solution'],
        num_rows: 75197
    })
})

=== Example 1 ===
{'lang': 'cpp', 'raw_index': 101533, 'index': 4626, 'seed': '  int n;\n  cin >> n;\n  vector<int> a(n + 1), b(n + 1);\n  for (int i = 1; i <= n; ++i) cin >> a[i] >> b[i];\n  auto c = convolution(a, b);\n  for (int i = 1; i <= 2 * n; ++i) cout << c[i] << endl;\n}\n', 'openai_fingerprint': 'fp_eeff13170a', 'problem': 'You are given two arrays, A and B, each of length n. You need to perform a convolution operation on these arrays and output the resulting array.\n\nThe convolution of two arrays A and B is defined as follows:\n- Let C be the resulting array of length 2n-1, where C[i] = Σ(A[j] * B[i-j]) for j = max(0, i-n+1) to min(i, n-1).\n\nWrite a function or method to perform the convolution operation and return the resulting array C.\n\nFunction Signature: \n```cpp\nvector<int> convolution(vector<i

In [ ]:
def format_prompt(example):
    return {
        "text": f"""### Instruction:
{example['problem']}

### Response:
{example['solution']}"""
    }

# Apply the formatting to the full dataset
formatted_dataset = dataset['train'].map(format_prompt)

# Drop columns we don't need
formatted_dataset = formatted_dataset.remove_columns(
    ['lang', 'raw_index', 'index', 'seed', 'openai_fingerprint', 'problem', 'solution']
)

print(formatted_dataset)
print("\n=== Formatted Example ===")
print(formatted_dataset[0]['text'])

In [ ]:
# Split into 95% train, 5% validation
split_dataset = formatted_dataset.train_test_split(test_size=0.05, seed=42)

train_dataset = split_dataset['train']
val_dataset = split_dataset['test']

print(f"Training examples: {len(train_dataset)}")
print(f"Validation examples: {len(val_dataset)}")

In [ ]:
# Save to disk so you can reload it instantly next session
train_dataset.save_to_disk("/content/cobble_train")
val_dataset.save_to_disk("/content/cobble_val")

print("Datasets saved successfully")
print(f"Train: {len(train_dataset)} examples")
print(f"Val: {len(val_dataset)} examples")